# Inferencia Estadística Multigrupo: Análisis de Varianza (ANOVA de 1 factor) y Pruebas Post-Hoc (Tukey)

#Tarea 1

In [11]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Datos empíricos (Consumo de RAM en MB)
np.random.seed(42)
memoria_algo_A = np.random.normal(loc=120.5, scale=12.0, size=35)
memoria_algo_B = np.random.normal(loc=128.2, scale=15.0, size=40)

alpha = 0.05

# Ejecución de la prueba T para muestras independientes (2 colas)
stat_ind, p_val_ind = ttest_ind(memoria_algo_A, memoria_algo_B, equal_var=True)

print("--- A/B Testing: Algoritmo A vs Algoritmo B ---")
print(f"Media Algo A: {np.mean(memoria_algo_A):.2f} MB | Media Algo B: {np.mean(memoria_algo_B):.2f} MB")
print(f"Estadístico T: {stat_ind:.4f}")
print(f"Valor-p: {p_val_ind:.4e}")

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Datos empíricos (Consumo de RAM en MB)
np.random.seed(42)
memoria_algo_A = np.random.normal(loc=120.5, scale=12.0, size=35)
memoria_algo_B = np.random.normal(loc=128.2, scale=15.0, size=40)

alpha = 0.05

# Ejecución de la prueba T para muestras independientes (2 colas)
stat_ind, p_val_ind = ttest_ind(memoria_algo_A, memoria_algo_B, equal_var=True)

print("--- A/B Testing: Algoritmo A vs Algoritmo B ---")
print(f"Media Algo A: {np.mean(memoria_algo_A):.2f} MB | Media Algo B: {np.mean(memoria_algo_B):.2f} MB")
print(f"Estadístico T: {stat_ind:.4f}")
print(f"Valor-p: {p_val_ind:.4e}")


--- A/B Testing: Algoritmo A vs Algoritmo B ---
Media Algo A: 118.91 MB | Media Algo B: 126.71 MB
Estadístico T: -2.5362
Valor-p: 1.3347e-02
--- A/B Testing: Algoritmo A vs Algoritmo B ---
Media Algo A: 118.91 MB | Media Algo B: 126.71 MB
Estadístico T: -2.5362
Valor-p: 1.3347e-02


#Tarea 2

In [ ]:
from scipy.stats import ttest_rel

# Latencia en ms (las posiciones en los arrays corresponden al mismo servidor)
latencia_antes = np.array([45, 52, 48, 55, 60, 42, 49, 58, 51, 46, 50, 47, 53, 59, 44])
latencia_despues = np.array([41, 50, 45, 50, 56, 40, 46, 53, 48, 42, 47, 45, 51, 55, 40])

# Prueba pareada (H1: la latencia "después" es menor, por tanto (antes - despues) > 0)
# Para evaluar si "después" bajó, usamos una prueba de cola superior sobre las diferencias (antes - despues)
# SciPy modernos permiten alternative='greater' en ttest_rel
stat_rel, p_val_rel = ttest_rel(latencia_antes, latencia_despues, alternative='greater')

print("\n--- Análisis Pareado: Impacto del Nuevo Firewall ---")
print(f"Media Diferencias (Antes - Después): {np.mean(latencia_antes - latencia_despues):.2f} ms")
print(f"Estadístico T Pareado: {stat_rel:.4f}")
print(f"Valor-p (1 cola): {p_val_rel:.4e}")

if p_val_rel < 0.05:
    print("Conclusión: Se RECHAZA H0. El firewall redujo significativamente la latencia.")
else:
    print("Conclusión: NO se rechaza H0. El firewall no mejoró la latencia.")


--- Análisis Pareado: Impacto del Nuevo Firewall ---
Media Diferencias (Antes - Después): 3.33 ms
Estadístico T Pareado: 12.3359
Valor-p (1 cola): 3.2787e-09
Conclusión: Se RECHAZA H0. El firewall redujo significativamente la latencia.


#Tarea 3

In [17]:
import pandas as pd
from scipy import stats

nombres_columnas = ["Fecha", "Tipo", "Emisor", "Cantidad", "Valor_Nominal",
                    "Precio", "Valor_Nominal_Total", "Monto_Efectivo",
                    "Casa_Origen", "Casa_Destino", "Bolsa"]

df = pd.read_csv("BVG_Acciones_export.csv", names=nombres_columnas, skiprows=1)

# Variable categórica 'Fecha' para extraer el mes
df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
df['Mes_Num'] = df['Fecha'].dt.month

# Variable cuantitativa 'Precio' para los dos grupos (Enero vs Julio)
grupo1 = df[df['Mes_Num'] == 1]['Precio'].dropna() # Enero
grupo2 = df[df['Mes_Num'] == 7]['Precio'].dropna() # Julio

# A/B Test (t-test para muestras independientes)
t_stat, p_value = stats.ttest_ind(grupo1, grupo2, nan_policy='omit')

# Resultados
print(f"Estadístico t: {t_stat:.4f}")
print(f"Valor p: {p_value:.4f}")


Estadístico t: 0.7448
Valor p: 0.4565


#Tarea 4

In [16]:
import numpy as np
from scipy.stats import levene, ttest_ind

# Uso de la Tarea 1
np.random.seed(42)
memoria_algo_A = np.random.normal(loc=120.5, scale=12.0, size=35)
memoria_algo_B = np.random.normal(loc=128.2, scale=15.0, size=40)

# Test de Levene (H0: Varianzas iguales)
stat_levene, p_val_levene = levene(memoria_algo_A, memoria_algo_B)

print(f"--- Uso de Levene ---")
print(f"Estadístico de Levene: {stat_levene:.4f}")
print(f"Valor-p de Levene: {p_val_levene:.4f}")

# Si p < 0.05, rechazamos la igualdad de varianzas (Heterocedasticidad)
if p_val_levene < 0.05:
    decision = "Heterocedasticidad detectada"
    metodo = "Prueba T de Welch (varianzas desiguales)"
    # equal_var = False
    t_stat, p_val_final = ttest_ind(memoria_algo_A, memoria_algo_B, equal_var=False)
else:
    decision = "Homocedasticidad confirmada"
    metodo = "Prueba T de Student clásica (varianzas iguales)"
    # equal_var = True
    t_stat, p_val_final = ttest_ind(memoria_algo_A, memoria_algo_B, equal_var=True)

print(f"\nResultado: {decision}")
print(f"Por lo tanto se usa el método: {metodo}")
print("-" * 40)
print(f"Estadístico T final: {t_stat:.4f}")
print(f"Valor-p final: {p_val_final:.4e}")

--- Uso de Levene ---
Estadístico de Levene: 3.0700
Valor-p de Levene: 0.0840

Resultado: Homocedasticidad confirmada
Por lo tanto se usa el método: Prueba T de Student clásica (varianzas iguales)
----------------------------------------
Estadístico T final: -2.5362
Valor-p final: 1.3347e-02


#Preguntas de control

# **1. ¿Por qué es un error metodológico grave aplicar una prueba T para muestras independientes (Tarea 1) en el escenario de la Tarea 2 (evaluación antes/después del firewall en los mismos servidores)? Explique el impacto en el término de covarianza.**

##Es un error usar la prueba T para muestras independientes porque las mediciones antes y después del firewall corresponden a los mismos servidores, por lo que los datos están relacionados. Al asumir que son independientes, se ignora la relación (covarianza) entre las mediciones y se pueden obtener resultados y conclusiones estadísticas incorrectas. Por ello, en este caso debe utilizarse la prueba T para muestras pareadas

# **2.Describa matemáticamente y gráficamente qué es la "Prueba T de Welch". ¿Qué parámetro fundamental (calculado a partir de  y la varianza) ajusta esta prueba para evitar cometer Errores Tipo I al comparar distribuciones muy dispares?**
## Se puede ver mejor la respuesta en el PDF adjunto en el envio.

# **3.En el análisis de su Proyecto Integrador (Tarea 3), si el Intervalo de Confianza para la diferencia de medias (u1 - u2) incluyera al número cero (0), ¿cuál sería inevitablemente el resultado de su prueba de hipótesis usando a = 0.05?**
## Falta de responder

# **4. Según lo investigado en la Tarea 4 (ABI), si el Test de Levene arroja un valor -p = 0.85, ¿cuál es la conclusión estadística respecto a las varianzas y qué versión del Test T debería usar?**
## Falta de responder

# **5. En el mundo real del Data Science, ¿cómo difiere el concepto estadístico clásico enseñado hoy frente a la ejecución comercial de un "A/B Testing" en plataformas web (ej. comparar el color de un botón de compra)?**

## El concepto clásico de la estadística busca certeza científica absoluta, mientras que la ejecución comercial busca velocidad para la toma de decisiones de negocio. Las diferencias clave son:
## Peeking: Mientras que la estadística tradicional prohíbe mirar los datos antes de tiempo para evitar falsos positivos, el negocio exige ver resultados diarios. Por esto, la industria web está reemplazando el enfoque clásico (Frecuentista) por la Estadística Bayesiana y algoritmos adaptativos (Multi-Armed Bandits), que permiten tomar decisiones sobre la marcha de forma matemáticamente válida.
## El tamaño de la muestra: En la teoría se esperan meses para acumular muestras gigantescas que detecten cambios mínimos. En el comercio digital, el costo de oportunidad de tener una web en espera es demasiado alto. El Data Scientist comercial ajusta el diseño del test para detectar solo efectos grandes, cerrando los experimentos rápido si no mueven la aguja del negocio.
## Efecto Novedad y Ciclos: Los modelos teóricos asumen datos estables y homogéneos. En la web, por ejemplo, cambiar el color de un botón genera un pico artificial de clics solo por ser algo novedoso. Para limpiar este sesgo y promediar los hábitos cambiantes de los usuarios (que se comportan distinto un martes frente a un domingo), los experimentos reales se obligan a correr en ciclos cerrados de semanas completas (7 o 14 días).
